# ⚡ Caminho Mínimo — Dijkstra

O **Algoritmo de Dijkstra** resolve caminhos mínimos de uma única origem em grafos com **pesos não negativos**. Mantém um conjunto $S$ de vértices com distâncias definitivas e cresce $S$ escolhendo sempre o vértice com menor estimativa atual na fronteira (estrutura: **fila de prioridade/heap**).

## 🎯 Ideia Geral
- Inicializa $p[s]=0$ e $p[v]=+\infty$ para $v\ne s$; $antecessor[v]=\text{NULL}$.
- Enquanto houver vértices fora de $S$: extrai o vértice $u$ com menor $p[u]$ do heap e relaxa cada aresta $(u,v)$.
- Relaxamento: se $p[v] > p[u] + w(u,v)$, atualiza $p[v]$ e $antecessor[v] = u$.
- Como todos os pesos são não negativos, quando $u$ sai do heap sua distância é definitiva.

## ⚠️ Restrições
- Requer pesos de arestas **não negativos**.
- Com pesos negativos, use **Bellman-Ford**.

In [ ]:
from math import inf
import heapq
from typing import Any, Dict, Tuple, List, Optional
import networkx as nx
import matplotlib.pyplot as plt

def dijkstra(G: nx.Graph, s: Any, weight: str = 'weight') -> Tuple[Dict[Any,float], Dict[Any,Optional[Any]]]:
    """
    Implementa Dijkstra usando heap (min-heap).
    Retorna (p, antecessor) com p[v] = distância mínima estimada de s a v.
    Suporta nx.Graph e nx.DiGraph. Requer pesos não negativos.
    """
    # Verificação leve de pesos negativos
    for u, v, data in G.edges(data=True):
        if float(data.get(weight, 1.0)) < 0:
            raise ValueError('Dijkstra requer pesos não negativos')

    p: Dict[Any,float] = {v: inf for v in G.nodes()}
    pai: Dict[Any,Optional[Any]] = {v: None for v in G.nodes()}
    p[s] = 0.0

    heap: List[Tuple[float, Any]] = [(0.0, s)]
    visitado = set()  # S

    while heap:
        dist_u, u = heapq.heappop(heap)
        if u in visitado:
            continue
        visitado.add(u)
        # relaxa as arestas que saem de u
        for v, data in G[u].items():
            w = float(data.get(weight, 1.0))
            if p[u] + w < p[v]:
                p[v] = p[u] + w
                pai[v] = u
                heapq.heappush(heap, (p[v], v))

    return p, pai

def reconstruir_caminho(pai: Dict[Any,Any], s: Any, v: Any) -> Optional[List[Any]]:
    if v == s:
        return [s]
    if pai.get(v) is None:
        return None
    caminho = [v]
    while v != s:
        v = pai[v]
        caminho.append(v)
    caminho.reverse()
    return caminho

def arvore_caminhos_minimos(G: nx.Graph, pai: Dict[Any,Any]) -> nx.DiGraph:
    T = nx.DiGraph()
    T.add_nodes_from(G.nodes())
    for v,u in pai.items():
        if u is not None:
            T.add_edge(u, v)
    return T

## 🧪 Exemplo 1 — Grafo ponderado (sem pesos negativos)

In [ ]:
G = nx.Graph()
G.add_weighted_edges_from([
    ('A','B',4), ('A','H',8), ('B','H',11), ('B','C',8), ('C','I',2), ('C','F',4),
    ('C','D',7), ('D','E',9), ('D','F',14), ('E','F',10), ('H','I',7), ('H','G',1),
    ('G','I',6), ('G','F',2)
])
p, pai = dijkstra(G, 'A')
print('Distâncias de A:', {k: round(v,1) for k,v in p.items()})

# Verificação com NetworkX
try:
    dist_nx, pred_nx = nx.single_source_dijkstra(G, 'A')
    print('Distâncias NetworkX:', {k: round(v,1) for k,v in dist_nx.items()})
except Exception as e:
    print('single_source_dijkstra indisponível:', e)

T = arvore_caminhos_minimos(G, pai)
pos = nx.spring_layout(G, seed=5)
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=900, font_size=12)
labels = {(u,v): f'{w:.1f}' for u,v,w in G.edges(data='weight')}
nx.draw_networkx_edge_labels(G, pos, edge_labels=labels, font_color='darkgreen')
plt.title('Grafo ponderado')

plt.subplot(1,2,2)
nx.draw(G, pos, with_labels=True, node_color='lightgreen', node_size=900, font_size=12)
nx.draw_networkx_edges(G, pos, edgelist=T.edges(), edge_color='crimson', width=3, arrows=True, arrowsize=20)
plt.title('Árvore de caminhos mínimos (a partir de A)')
plt.tight_layout()
plt.show()

## ⏱️ Complexidade
- Inicialização: O(|V|).
- Cada extração do heap: O(log |V|), total de |V| vezes → O(|V| log |V|).
- Relaxamento de arestas: cada atualização empilha no heap (decrease-key implícito), total O(|E| log |V|).
- Total: **O(|E| log |V|)**.

## 🧩 Exercícios
1. Gere grafos aleatórios sem pesos negativos e compare as distâncias com Bellman-Ford (devem coincidir).
2. Conte operações no heap (push/pop) e compare para diferentes densidades de grafo.
3. Teste casos com vértices inalcançáveis: o valor deve permanecer +∞ e o predecessor NULL.
4. Reconstrua caminhos de s até todos os v alcançáveis e valide com NetworkX.
5. Mostre empiricamente o impacto de pesos muito discrepantes na forma da árvore de caminhos mínimos.